# Most-Frequent Words: The Signal Everyone Discards

**Computational Text Analysis Pack · Module 09 — Stylometry**

*Technique:* MFW feature sets; why function words identify authors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/07-stylometry/09a_most_frequent_words.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Every method so far removed function words as noise. Stylometry inverts that
completely: **the function words are the signal.**

The reason is that *the*, *of*, *and*, *but* are used unconsciously. An author
cannot easily change how often they write *of* — it is not a stylistic choice,
it is a habit — and the rate is remarkably stable across a writer's work and
remarkably different between writers.

Content words tell you the subject. Function words tell you the author.

## The tooling

**numpy** for the z-scored feature matrices and **scipy** for clustering.
Stylometry's field standard is **stylo** (R), which implements every Delta
variant plus bootstrap consensus trees; **faststylometry** is the Python
equivalent. We write Delta out because it is ten lines.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# Group the colonial corpus by work, then keep authors with several works
# so the question "does this author's work resemble itself?" is answerable.
works = (df_colonial.groupby(["work_id", "title", "author"])["text"]
                    .apply(" ".join).reset_index())
works["n_words"] = works["text"].str.split().str.len()
works = works[works["n_words"] >= 5000].reset_index(drop=True)

counts = works["author"].value_counts()
multi = counts[counts >= 2].index.tolist()
print(f"{len(works)} works, {works['author'].nunique()} authors")
print(f"{len(multi)} authors with 2+ works in the corpus")
print()
print(counts.head(10).to_string())

## The MFW feature set

Take the *n* most frequent words **across the whole corpus**, then measure each
document's rate for each of them. That is the entire feature set — no TF-IDF, no
topic model, and deliberately no content.

In [ ]:
MFW = 150                   # ← change this

corpus_counts = Counter()
tokenised = []
for t in works["text"]:
    toks = tokenise(t)
    tokenised.append(toks)
    corpus_counts.update(toks)

features = [w for w, _ in corpus_counts.most_common(MFW)]
print(f"the {MFW} most frequent words in the corpus:")
print("   " + ", ".join(features[:40]) + " …")
print()

rates = np.array([[toks.count(w) / len(toks) * 100 for w in features]
                  for toks in tokenised])
print(f"feature matrix: {rates.shape[0]} works × {rates.shape[1]} words (rates per 100)")

In [ ]:
# How stable is a function word within an author, compared to between authors?
WORD = "the"                # ← change this
j = features.index(WORD) if WORD in features else 0

work_rates = pd.DataFrame({"author": works["author"], "rate": rates[:, j]})
grp = work_rates[work_rates["author"].isin(multi)].groupby("author")["rate"]
summary = grp.agg(works="size", mean="mean", spread=lambda s: s.max() - s.min())
print(f"rate of '{features[j]}' per 100 words")
print(summary.sort_values("mean", ascending=False).head(10)
      .to_string(float_format=lambda v: f"{v:.2f}"))
print()
within = grp.std().mean()
between = work_rates["rate"].std()
print(f"typical spread WITHIN an author : {within:.3f}")
print(f"spread ACROSS the whole corpus  : {between:.3f}")
print()
print("When the second is comfortably larger than the first, the word carries")
print("authorial signal. That ratio is the whole basis of stylometry.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
show = [a for a in multi[:10]]
data = [work_rates[work_rates["author"] == a]["rate"].values for a in show]
ax.boxplot(data, vert=False, showfliers=True)
ax.set_yticks(range(1, len(show) + 1))
ax.set_yticklabels([a[:28] for a in show])
ax.set_xlabel(f"rate of '{features[j]}' per 100 words")
ax.set_title("Function-word rates cluster by author")
plt.tight_layout(); plt.show()

### 🔧 Your turn 1

Change `WORD` to `"of"`, `"and"`, `"but"`, `"which"`.

Some separate authors well and some do not. Stylometry uses a hundred or more
together precisely because no single word is reliable — the pattern across many
is what identifies a writer.

## Why content words are the wrong features here

Swap in the most *distinctive* words instead of the most frequent, and the
grouping stops being about authorship and starts being about subject.

In [ ]:
content = [w for w, _ in corpus_counts.most_common(2000)][400:550]
c_rates = np.array([[toks.count(w) / len(toks) * 100 for w in content]
                    for toks in tokenised])

def separation(matrix, labels):
    """Ratio of between-author spread to within-author spread. Higher is better."""
    z = (matrix - matrix.mean(0)) / (matrix.std(0) + 1e-9)
    frame = pd.DataFrame(z)
    frame["a"] = labels.values
    sub = frame[frame["a"].isin(multi)]
    within = sub.groupby("a").std(numeric_only=True).mean().mean()
    between = sub.drop(columns="a").std().mean()
    return between / max(within, 1e-9)

print(f"function words (top {MFW})   separation ratio {separation(rates, works['author']):.2f}")
print(f"content words (ranks 400-550) separation ratio {separation(c_rates, works['author']):.2f}")
print()
print("Function words win. They are also robust to genre and subject in a way")
print("content words never are — which is what makes attribution possible at all.")

### 🔧 Your turn 2

Set `MFW` to 50, then 500, and re-run the separation comparison.

Too few words and you have not enough signal; too many and you start including
content words, which drags the subject back in. The stylometry literature
converges on 100–300 for exactly this reason.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** *Of* and *the* are usually the strongest single separators in
English, because their rates vary with syntactic habits — nominal versus verbal
style. *But* and *which* are noisier: they vary with genre and with the
particular argument being made. No single word does the job, which is the
argument for the multivariate methods in 09b.

**Your turn 2.** The separation ratio typically rises steeply to about 100 words,
plateaus, then declines slowly as content words enter. Report the MFW count you
used — results are sensitive to it, and a claim that only holds at one specific
value is not a claim.

**One warning for this corpus specifically.** These are Project Gutenberg
transcriptions, and OCR and transcription conventions differ between them.
Apparent stylistic differences can be differences in how a text was digitised.
Before attributing anything, check that your candidate texts came from comparable
sources.

</details>

## Where this stops

You have the feature set. Next: **09b — Burrows's Delta**, the measure that turns
it into an attribution.